# Rossman dataset 

Fuente XGBoost: https://xgboost.readthedocs.io/

https://www.kaggle.com/competitions/rossmann-store-sales/data

- Id - an Id that represents a (Store, Date) duple within the test set
- Store - a unique Id for each store
- Sales - the turnover for any given day (this is what you are predicting)
- Customers - the number of customers on a given day
- Open - an indicator for whether the store was open: 0 = closed, 1 = open
- StateHoliday - indicates a state holiday. Normally all stores, with few exceptions, are closed on state holidays. Note that all schools are closed on public holidays and weekends. a = public holiday, b = Easter holiday, c = Christmas, 0 = None
- SchoolHoliday - indicates if the (Store, Date) was affected by the closure of public schools
- StoreType - differentiates between 4 different store models: a, b, c, d
- Assortment - describes an assortment level: a = basic, b = extra, c = extended
- CompetitionDistance - distance in meters to the nearest competitor store
- CompetitionOpenSince[Month/Year] - gives the approximate year and month of the time the nearest competitor was opened
- Promo - indicates whether a store is running a promo on that day
- Promo2 - Promo2 is a continuing and consecutive promotion for some stores: 0 = store is not participating, 1 = store is participating
- Promo2Since[Year/Week] - describes the year and calendar week when the store started participating in Promo2
- PromoInterval - describes the consecutive intervals Promo2 is started, naming the months the promotion is started anew. E.g. "Feb,May,Aug,Nov" means each round starts in February, May, August, November of any given year for that store

## Versión con XGBoost

Esta versión conserva el flujo del X-learner de Rossmann y usa XGBoost en los modelos de resultados, pseudo-efectos y propensity score. Las categorías se representan con variables dummy, como en el notebook original.


In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# ----------------------------
# CARGA Y PREPROCESAMIENTO
# ----------------------------
# Carga de datos
train = pd.read_csv('rossman.csv')
stores = pd.read_csv('store.csv')

# Merge de ambos datasets
df = pd.merge(train, stores, on='Store')

# Convertir la columna de fecha
df['Date'] = pd.to_datetime(df['Date'])

# Ordenar por fecha para evitar leakage
df = df.sort_values('Date')

# Filtrar solo tiendas abiertas
df = df[df['Open'] == 1]

# ----------------------------
# FEATURE ENGINEERING
# ----------------------------

# Variables temporales útiles
df['month'] = df['Date'].dt.month
df['day_of_month'] = df['Date'].dt.day
df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)

# Eliminar columnas que no usaremos
df = df.drop(columns=['Date', 'Store', 'Customers', 'Open', 'Promo2'])



/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_60000/1654015796.py:11: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('rossman.csv')


### ⚙️ Parámetros principales de XGBoost

- `n_estimators`: número máximo de árboles.
- `max_depth`: profundidad máxima de cada árbol.
- `learning_rate`: tamaño del paso de cada árbol.
- `subsample`: fracción de filas usada por árbol.
- `colsample_bytree`: fracción de variables usada por árbol.
- `min_child_weight`, `gamma`, `reg_alpha` y `reg_lambda`: controlan la complejidad y regularización.
- `tree_method="hist"`: construcción eficiente de árboles mediante histogramas.

Los mismos tipos de modelos se usan en cada etapa del X-learner: regresores para resultados y pseudo-efectos, y un clasificador para el propensity score.


## Conditional average treatment effect (CATE)

Para una covariable \( X = x \), el CATE se define como:

$$
\text{CATE}(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x]
$$

> Es decir, **el efecto esperado del tratamiento para individuos con características \( X = x \)**.


# X-learner

El objetivo es estimar el efecto causal heterogéneo:

$$
\tau(x) = E[Y(1) - Y(0) \mid X = x]
$$

donde:
- $Y(1)$: outcome si $T=1$  
- $Y(0)$: outcome si $T=0$  
- $X$: covariables  
- $T \in \{0,1\}$: tratamiento  

El problema es que **no observamos ambos $Y(1)$ y $Y(0)$ para el mismo individuo**.

---

## Paso 1: Modelos base (tipo T-learner)

Estimamos:

$$
m_1(x) = E[Y \mid X = x, T = 1]
$$

$$
m_0(x) = E[Y \mid X = x, T = 0]
$$

---

## Paso 2: Construcción de pseudo-efectos

### Para $T=1$:

Observamos $Y(1)$ pero no $Y(0)$:

$$
\tilde{\tau}^{(1)} = Y - m_0(X)
$$

Interpretación:
> lo que pasó menos lo que hubiera pasado si $T=0$

---

### Para $T=0$:

Observamos $Y(0)$ pero no $Y(1)$:

$$
\tilde{\tau}^{(0)} = m_1(X) - Y
$$

Interpretación:
> lo que hubiera pasado si $T=1$ menos lo que pasó

---

## Paso 3: Modelos sobre los pseudo-efectos

$$
\tau_1(x) = E[\tilde{\tau}^{(1)} \mid X = x]
$$

$$
\tau_0(x) = E[\tilde{\tau}^{(0)} \mid X = x]
$$

---

## Paso 4: Propensity score

$$
e(x) = P(T = 1 \mid X = x)
$$

---

## Paso 5: Combinación final

$$
\tau(x) = e(x)\,\tau_0(x) + (1 - e(x))\,\tau_1(x)
$$

---

## Intuición clave

- $\tau_1(x)$ se aprende de observaciones con $T=1$  
- $\tau_0(x)$ se aprende de observaciones con $T=0$  

El peso es cruzado:

- Si $e(x)$ es alto (muchos $T=1$)  
  → se confía más en $\tau_0(x)$  

- Si $e(x)$ es bajo  
  → se confía más en $\tau_1(x)$  

---

## Resumen compacto

$$
\text{X-learner} =
\begin{cases}
\text{Estimar } m_0, m_1 \\
\text{Construir pseudo-efectos} \\
\text{Modelar } \tau_0, \tau_1 \\
\text{Combinar con } e(x)
\end{cases}
$$

---

## Comentarios prácticos

- Funciona bien cuando hay desbalance entre $T=1$ y $T=0$
- Depende fuertemente de $m_0(x)$ y $m_1(x)$
- No es ortogonal (vs enfoques tipo Double ML)
- Es flexible: puedes usar GBM, RF, etc.

---

## Limitación importante

Aplica cuando:

$$
T \in \{0,1\}
$$

Para tratamiento continuo (ej. monto de crédito), necesitas otros enfoques.

## Paso 1: T-learner


In [2]:
# ----------------------------
# SEPARACIÓN TEMPORAL: 80% pasado, 20% futuro
# ----------------------------
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Separar X y y
target = 'Sales'
X_train = train_df.drop(columns=target)
y_train = train_df[target]
X_test = test_df.drop(columns=target)
y_test = test_df[target]

# Guardar columna Promo antes de get_dummies
promo_train = X_train['Promo']
promo_test = X_test['Promo']

# Codificar variables categóricas
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Alinear columnas
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train.columns = X_train.columns.str.replace(r'[^\w]', '_', regex=True)
X_test.columns = X_test.columns.str.replace(r'[^\w]', '_', regex=True)

# ----------------------------
# SEPARAR POR PROMO (entrenamiento y prueba)
# ----------------------------

# Detectar la columna codificada de Promo
promo_col = [col for col in X_train.columns if 'Promo' in col][0]  # asume binaria

# Train: Promo = 1
X_train_promo1 = X_train[X_train[promo_col] == 1]
y_train_promo1 = y_train.loc[X_train_promo1.index]

# Train: Promo = 0
X_train_promo0 = X_train[X_train[promo_col] == 0]
y_train_promo0 = y_train.loc[X_train_promo0.index]

# Test: Promo = 1
X_test_promo1 = X_test[X_test[promo_col] == 1]
y_test_promo1 = y_test.loc[X_test_promo1.index]

# Test: Promo = 0
X_test_promo0 = X_test[X_test[promo_col] == 0]
y_test_promo0 = y_test.loc[X_test_promo0.index]

# ----------------------------
# CREAR VECTORES BINARIOS DE TRATAMIENTO
# ----------------------------
y_train_promo = promo_train.reset_index(drop=True)
y_test_promo = promo_test.reset_index(drop=True)


In [3]:
#Definir modelo sin promo
model_no_promo=XGBRegressor()
# Entrenar
model_no_promo.fit(X_train_promo0,y_train_promo0)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [4]:
# Definir modelo con promo
model_promo=XGBRegressor()
# Entrenar
model_promo.fit(X_train_promo1,y_train_promo1)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [5]:
# CATE con T learner
df_cate=pd.DataFrame({
    'promo':X_test['Promo'],
    'ventas':'y_test',
    'efecto_promo':model_promo.predict(X_test)- model_no_promo.predict(X_test)
})
df_cate.query("promo==1")

,promo,ventas,efecto_promo
203882,1,y_test,759.419434
203883,1,y_test,841.686523
203884,1,y_test,1910.139648
203885,1,y_test,30.178223
203886,1,y_test,2857.103516
...,...,...,...
745,1,y_test,2499.405273
746,1,y_test,3295.600586
747,1,y_test,2134.995117
741,1,y_test,1660.680664


# Parte 2

Entonces $D^0_i$ es una estimación del efecto causal individual para ese individuo, inferido desde el contrafactual no tratado.

In [6]:
# Imputar efectos individuales
# Lo que pasó menos lo que habría pasado sin tratamiento


Entonces $D^1_i$ también es una estimación del efecto causal individual, pero desde el lado opuesto.

In [7]:
# Lo que habría pasado con tratamiento menos lo que pasó


### Entrena los modelos

In [8]:
# Entrenar los modelos de efecto (CATEs)
# Tau0


# Tau1


# Parte 4 : Propensity score

In [9]:
# Eliminar columna dummy de Promo para entrenar el modelo
X_train_ps = X_train.drop(columns=[promo_col])
X_test_ps = X_test.drop(columns=[promo_col])

# Entrenar modelo de Propensity Score
ps_model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=42)
ps_model.fit(X_train_ps, y_train_promo)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=-1,
              num_parallel_tree=None, ...)

# Parte 5: Predecir 

In [10]:
tau1_preds = tau_model_1.predict(X_test)
tau0_preds = tau_model_0.predict(X_test)
propensity_score = ps_model.predict_proba(X_test_ps)[:, 1]

NameError: name 'tau_model_1' is not defined

In [ ]:
cate = propensity_score * tau1_preds + (1-propensity_score)*tau0_preds